# Step 25 — Direct tests of the mass account, the instrument's power, and lag robustness (CPU)

**Inputs.** Mount the outputs of **Step 19** (`step19_logits.npz`) and **Step 23** (`step23_committor.npz`). CPU only, about 1.5–2 hours.

1. **Direct reweighting test (reviewer concern 5).** Reweight the *data* reference to the model's class frequencies, $w_c=f^m_c/f^d_c$, and compare it pair by pair with the model's *raw* transitions and asymmetries. If geometry is right and only mass is wrong, the reweighted data should reproduce the raw model as closely as two data samples reproduce each other.
2. **The mass account in committor terms (concern 8).** Shift the classifier posterior by the model's class frequencies, $P_m(c\mid x)\propto P_d(c\mid x)\,f^m_c/f^d_c$, and recompute the committor-fidelity excess. If the excess in the decision window is mass, it should shrink.
3. **Power of the rebalanced test (concern 1).** Plant known *shape* errors in the data that leave class mass unchanged: within each class, reweight images by $e^{\beta u}$, where $u$ is the classifier margin on near-clean images, i.e. how prototypical an image is. Compare planted against clean data with exactly the statistics used for the model. This sweeps the size of the distortion (reported as the KL divergence of the reweighting), once for all classes and once for a single class.
4. **Lags across coordinates and thresholds (concern 3).** Decision levels at three cross-over thresholds; density transitions along three coordinates (raw log-odds, $\tanh(z/2)=2q-1$, signed square root). Rank correlations and lag factors for every combination.

Headline intervals use 1,000 class-block resamples.

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt, time
from itertools import combinations
from scipy.signal import fftconvolve
from scipy.stats import spearmanr, rankdata
plt.rcParams['figure.dpi'] = 120; rng = np.random.default_rng(25)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
kd = lambda p: 'va' if ((p[0] in VEH) != (p[1] in VEH)) else ('vv' if p[0] in VEH else 'aa')
def find(pat): p = glob.glob(f'/kaggle/input/**/{pat}', recursive=True); assert p, f'mount the output containing {pat}'; return p[0]
Z19 = np.load(find('step19_logits.npz')); L_TE, W_TE, L_TR, W_TR, L_M, W_M, SIG = [Z19[k] for k in ['L_TE', 'W_TE', 'L_TR', 'W_TR', 'L_M', 'W_M', 'SIG_REC']]
ZC = np.load(find('step23_committor.npz')); END, PAR, Q_CLF, LEV_SIG = ZC['END'], ZC['PAR'], ZC['Q_CLF'], ZC['LEV_SIG']; L_LEV, M_PAR, K_BR = END.shape
F_D = np.bincount(W_TE.argmax(1), minlength=10)/len(W_TE); F_M = np.bincount(W_M.argmax(1), minlength=10)/len(W_M)
print('data frequencies', np.round(F_D, 3)); print('model frequencies', np.round(F_M, 3), f'mean |log f_m/f_d| = {np.mean(np.abs(np.log(F_M/F_D))):.3f}')
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3, c1 + 3*c3*s**2 + 2*c2*s, b
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG))] for a, b in PAIRS}
def analyse(L, W, a, b, idx, balance=False, rw=None, g0=0, slope=False):
    """Along the pair log-odds (reference standardisation): returns dict(sigma, gi, h, a, b[, s]). rw: extra class reweighting factors (for part C)."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if rw is not None: Wa = Wa*rw[a]; Wb = Wb*rw[b]
    if balance: tot = 0.5*(Wa.sum()+Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    def dist(g):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: return None
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; return wkde_score((z - mid)/half, Wa + Wb)
    prev = None if g0 == 0 else 1
    for g in range(g0, len(SIG)):
        d = dist(g); nm = 1 if d is None else n_max(stationary(*d))
        if nm >= 2 and prev is not None:
            h, a_, b_ = cusp(*d); out = dict(sigma=float(SIG[g]), gi=g, h=h, a=a_, b=b_)
            if slope:                                            # a(log sigma) near the transition, from the levels just above and below
                gs = [k for k in range(max(0, g-4), min(len(SIG), g+3)) if dist(k) is not None]
                av = [cusp(*dist(k))[1] for k in gs]
                if len(gs) >= 3: out['s'] = -np.polyfit(np.log(SIG[gs]), av, 1)[0]
            return out
        if nm == 1: prev = 1
    return None
def boot(L, W, a, b, n, B, balance=False, ref=False, g_hint=None):
    """Bootstrap medians and 95% intervals of sigma_bif and h at sample size n."""
    ss, hs = [], []; g0 = 0 if g_hint is None else max(0, g_hint - 8)
    for _ in range(B):
        idx = rng.choice(len(W), size=n, replace=True); r = analyse(L, W, a, b, idx, balance, g0=g0)
        if r: ss.append(r['sigma']); hs.append(r['h'])
    if len(ss) < B//3: return None
    return dict(sigma=np.median(ss), s_lo=np.percentile(ss, 2.5), s_hi=np.percentile(ss, 97.5), h=np.median(hs), h_lo=np.percentile(hs, 2.5), h_hi=np.percentile(hs, 97.5), n_ok=len(ss))
def class_block_ci(x, y, pairs, B=5000, fn=None):
    """Class-block bootstrap: resample the 10 classes with replacement, weight each pair by the product of its classes' multiplicities."""
    P = np.array(pairs); out = []
    for _ in range(B):
        cls = rng.choice(10, 10, replace=True); w = np.array([np.sum(cls == a)*np.sum(cls == b) for a, b in P], float)
        if w.sum() < 6: continue
        if fn is None:
            mx, my = np.average(x, weights=w), np.average(y, weights=w); out.append(np.sum(w*(x-mx)*(y-my))/np.sqrt(np.sum(w*(x-mx)**2)*np.sum(w*(y-my)**2)))
        else: out.append(fn(w))
    out = np.array(out, float); out = out[np.isfinite(out)]
    return np.percentile(out, [2.5, 97.5]) if len(out) > 20 else np.array([np.nan, np.nan])

def bmed(L, W, a, b, n, B=100, balance=False, rw=None, pool=None):
    """bootstrap medians of (sigma_bif, h); pool restricts the rows that are resampled"""
    rows = np.arange(len(W)) if pool is None else pool
    r0 = analyse(L, W, a, b, rng.choice(rows, size=n, replace=len(rows) < n), balance, rw)
    if r0 is None: return None
    out = []
    for _ in range(B):
        r = analyse(L, W, a, b, rng.choice(rows, size=n, replace=True), balance, rw, g0=max(0, r0['gi'] - 8))
        if r: out.append((r['sigma'], r['h']))
    if len(out) < B//3: return None
    o = np.array(out); return np.median(o[:, 0]), np.median(o[:, 1])
def cb_corr(x, y, P, B=1000): return class_block_ci(np.asarray(x), np.asarray(y), P, B=B)
def cb_mean_ratio(lr, P, B=1000): return class_block_ci(None, None, P, B=B, fn=lambda w: np.exp(np.average(lr, weights=w)))
def cb_sd(lr, P, B=1000): return class_block_ci(None, None, P, B=B, fn=lambda w: np.sqrt(np.average((lr - np.average(lr, weights=w))**2, weights=w)))
print('ready')

## 1. Direct reweighting test

In [ ]:
RW = F_M/F_D; n = len(W_M); t0 = time.time(); D1 = {}
for p in PAIRS:
    m = bmed(L_M, W_M, *p, n); d = bmed(L_TE, W_TE, *p, n); drw = bmed(L_TE, W_TE, *p, n, rw=RW); tr = bmed(L_TR, W_TR, *p, min(n, len(W_TR)))
    if m and d and drw and tr: D1[p] = dict(m=m, d=d, drw=drw, tr=tr)
P1 = list(D1); print(f"{len(P1)} pairs ({(time.time()-t0)/60:.0f} min)")
hm = np.array([D1[p]['m'][1] for p in P1]); hd = np.array([D1[p]['d'][1] for p in P1]); hdrw = np.array([D1[p]['drw'][1] for p in P1]); htr = np.array([D1[p]['tr'][1] for p in P1])
lr_raw = np.array([np.log(D1[p]['m'][0]/D1[p]['d'][0]) for p in P1]); lr_rw = np.array([np.log(D1[p]['m'][0]/D1[p]['drw'][0]) for p in P1]); lr_dd = np.array([np.log(D1[p]['tr'][0]/D1[p]['d'][0]) for p in P1])
print(f"{'comparison':>48} {'corr h [95%]':>22} {'SD location error [95%]':>26}")
for nm, h1, h2, lr in [('model raw vs data', hm, hd, lr_raw), ('model raw vs data reweighted to model mass', hm, hdrw, lr_rw), ('data (train) vs data (test): the floor', htr, hd, lr_dd)]:
    c = cb_corr(h1, h2, P1); s = cb_sd(lr, P1)
    print(f"{nm:>48} {np.corrcoef(h1, h2)[0,1]:7.3f} [{c[0]:.2f}, {c[1]:.2f}] {lr.std():12.3f} [{s[0]:.3f}, {s[1]:.3f}]")
ci = cb_mean_ratio(lr_rw, P1); print(f"location ratio, model raw / reweighted data: median {np.exp(np.median(lr_rw)):.3f}, geometric mean {np.exp(lr_rw.mean()):.3f} [class-block 95% CI {ci[0]:.3f}, {ci[1]:.3f}]")

## 2. The mass account in committor terms

In [ ]:
Q_MOD = np.stack([np.stack([np.bincount(END[l, m], minlength=10)/K_BR for m in range(M_PAR)]) for l in range(L_LEV)])
Q_SH = Q_CLF*(F_M/F_D)[None, None, :]; Q_SH = Q_SH/Q_SH.sum(2, keepdims=True)
rs = np.random.default_rng(7)
def excess(Q, idx):
    out = []
    for l in range(L_LEV):
        tv = 0.5*np.abs(Q_MOD[l, idx] - Q[l, idx]).sum(1).mean()
        sim = np.mean([0.5*np.abs(np.bincount(rs.choice(10, K_BR, p=q/q.sum()), minlength=10)/K_BR - q).sum() for q in Q[l, idx]])
        out.append(tv - sim)
    return np.array(out)
res = {}
for nm, Q in [('data posterior', Q_CLF), ('posterior shifted by model mass', Q_SH)]:
    e0 = excess(Q, np.arange(M_PAR)); bs = np.array([excess(Q, rng.choice(M_PAR, M_PAR)) for _ in range(300)]); res[nm] = (e0, bs)
print(f"{'sigma':>7} {'excess, data posterior [95%]':>32} {'excess, mass-shifted posterior [95%]':>40} {'difference [95%]':>26}")
for l, s in enumerate(LEV_SIG):
    (e0, b0), (e1, b1) = res['data posterior'], res['posterior shifted by model mass']; dif = b0[:, l] - b1[:, l]
    print(f"{s:7.3f} {e0[l]:12.3f} [{np.percentile(b0[:, l], 2.5):+.3f}, {np.percentile(b0[:, l], 97.5):+.3f}] {e1[l]:16.3f} [{np.percentile(b1[:, l], 2.5):+.3f}, {np.percentile(b1[:, l], 97.5):+.3f}] {e0[l]-e1[l]:+10.3f} [{np.percentile(dif, 2.5):+.3f}, {np.percentile(dif, 97.5):+.3f}]")
print('raw excesses are shown (no floor subtraction): the floor cancels in the difference column.')

## 3. Power: plant shape errors that leave class mass unchanged

In [ ]:
lab = W_TE.argmax(1); Lm = L_TE[-1].astype(np.float64)
srt = np.sort(Lm, 1); u = Lm[np.arange(len(lab)), lab] - np.where(Lm[np.arange(len(lab)), lab] == srt[:, -1], srt[:, -2], srt[:, -1])   # margin on near-clean images
u = (u - u.mean())/u.std()
half = rng.permutation(len(W_TE)); A_, B_ = half[:len(half)//2], half[len(half)//2:]
def planted_weights(beta, classes=None):
    w = np.ones(len(W_TE))
    cls = range(10) if classes is None else classes
    for c in cls:
        j = lab == c; e = np.exp(beta*u[j]); w[j] = e/e.mean()                 # mean 1 within class: class mass unchanged
    return w
def kl_of(w, classes=None):
    cls = range(10) if classes is None else classes
    return np.mean([np.mean(w[lab == c]*np.log(w[lab == c])) for c in cls])
nP = len(B_); PP = [p for p in PAIRS]
base = {p: bmed(L_TE, W_TE, *p, nP, B=60, pool=A_) for p in PP}
print(f"{'planted error':>26} {'KL (nats/class)':>15} {'corr h vs clean [95%]':>24} {'SD loc. error':>14} {'mean ratio [95%]':>22} {'pairs |dh|>0.05':>16}")
for nm, betas, cls in [('all classes', [0.0, 0.25, 0.5, 1.0], None), ('deer only', [0.5, 1.0, 2.0], [4])]:
    for beta in betas:
        w = planted_weights(beta, cls); Wp = W_TE*w[:, None]; rows = []
        for p in PP:
            r = bmed(L_TE, Wp, *p, nP, B=60, balance=True, pool=B_)
            if r and base[p]: rows.append((p, r))
        P_ = [x[0] for x in rows]; hp = np.array([x[1][1] for x in rows]); hb = np.array([base[p][1] for p in P_]); lr = np.array([np.log(x[1][0]/base[x[0]][0]) for x in rows])
        c = cb_corr(hp, hb, P_, B=500); mr = cb_mean_ratio(lr, P_, B=500)
        print(f"{nm + f', beta={beta}':>26} {kl_of(w, cls):15.4f} {np.corrcoef(hp, hb)[0,1]:9.3f} [{c[0]:.2f}, {c[1]:.2f}] {lr.std():14.3f} {np.exp(lr.mean()):9.3f} [{mr[0]:.2f}, {mr[1]:.2f}] {int(np.sum(np.abs(hp-hb) > 0.05)):>10}/{len(P_)}")
print("beta=0 is the split-half noise floor. A test with power shows correlation dropping, location scatter rising, or a ratio away from 1 as KL grows.")

## 4. Lag factors across coordinates and thresholds

In [ ]:
def crossover(a, b, J, min_count=20):
    X = np.full(L_LEV, np.nan)
    for l in range(L_LEV):
        same = sum(np.sum(END[l, j] == PAR[j]) for j in J); cross = sum(np.sum(END[l, j] == (b if PAR[j] == a else a)) for j in J)
        if same + cross >= min_count: X[l] = cross/(same + cross)
    return X
def decide(X, thr):
    ok = np.isfinite(X); good = ok & (X <= thr)
    for l in range(L_LEV):
        if good[l] and all(good[k] or not ok[k] for k in range(l, L_LEV)):
            if l > 0 and ok[l-1] and X[l-1] > thr:
                w = (X[l-1] - thr)/(X[l-1] - X[l]); return float(np.exp(np.log(LEV_SIG[l-1]) + w*(np.log(LEV_SIG[l]) - np.log(LEV_SIG[l-1]))))
            return float(LEV_SIG[l])
    return np.nan
WARPS = {'raw log-odds': lambda z: z, 'tanh(z/2) = 2q-1': lambda z: np.tanh(z/2), 'signed sqrt': lambda z: np.sign(z)*np.sqrt(np.abs(z))}
def sigma_bif_warp(L, W, a, b, warp, n=6000):
    idx = rng.choice(len(W), size=min(n, len(W)), replace=False); Wa, Wb = W[idx, a].astype(np.float64), W[idx, b].astype(np.float64); prev = None
    for g in range(len(SIG)):
        z = warp(L[g, idx, a].astype(np.float64) - L[g, idx, b]); mid, half, sep = std_stats(z, Wa + 1e-12, Wb + 1e-12)
        if sep < 0.3: prev = 1; continue
        sm, p = wkde_score((z - mid)/half, Wa + Wb); nm = n_max(stationary(sm, p))
        if nm >= 2 and prev is not None: return float(SIG[g])
        if nm == 1: prev = 1
    return None
J = {p: np.where(np.isin(PAR, list(p)))[0] for p in PAIRS}
DEC = {thr: {p: decide(crossover(*p, J[p]), thr) for p in PAIRS if len(J[p]) >= 20} for thr in [0.05, 0.1, 0.2]}
SB = {nm: {p: sigma_bif_warp(L_M, W_M, *p, f) for p in PAIRS} for nm, f in WARPS.items()}
print(f"{'decision threshold':>18} {'coordinate':>18} {'Spearman [95%]':>20} {'lag va':>8} {'lag vv':>8} {'lag aa':>8} {'lag all':>8}")
for thr, dec in DEC.items():
    for nm, sb in SB.items():
        P_ = [p for p in dec if np.isfinite(dec[p]) and sb.get(p)]; d = np.array([dec[p] for p in P_]); s = np.array([sb[p] for p in P_])
        rx, ry = rankdata(d), rankdata(s); c = cb_corr(rx, ry, P_, B=500); lag = d/s
        lk = {k: np.median(lag[[i for i, p in enumerate(P_) if kd(p) == k]]) for k in ['va', 'vv', 'aa']}
        print(f"{thr:18.2f} {nm:>18} {spearmanr(d, s).correlation:8.2f} [{c[0]:.2f}, {c[1]:.2f}] {lk['va']:8.2f} {lk['vv']:8.2f} {lk['aa']:8.2f} {np.median(lag):8.2f}")
print(f"committor grid spacing: median ratio between adjacent levels = {np.median(LEV_SIG[:-1]/LEV_SIG[1:]):.2f}")
for thr, dec in DEC.items():
    v = {k: np.median([dec[p] for p in dec if kd(p) == k and np.isfinite(dec[p])]) for k in ['va', 'vv', 'aa']}
    print(f"decision medians at threshold {thr}: va {v['va']:.2f}, vv {v['vv']:.2f}, aa {v['aa']:.2f};  va/aa = {v['va']/v['aa']:.2f}")

## What to report back
- **Part 1:** the three comparison lines and the ratio line.
- **Part 2:** the excess table.
- **Part 3:** the power table.
- **Part 4:** the threshold × coordinate table, the grid-spacing line, and the decision-median lines.